# Linear regression

This file experiments with linear regression to predict the target variables: total area burned, average daily area burned, and duration.  5-fold CV was conducted in order to compute performance metrics for full models and backward selected models.

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# import feature data
# data = pd.read_csv('../data/processed_data/Wildfire_Weather_2020_2024.csv')

In [ ]:
# data

In [ ]:
# data.shape

In [ ]:
# Keep only the first occurrence of each unique fire_id
# data = data.drop_duplicates(subset='unique_id', keep='first')

In [ ]:
# data.shape

In [ ]:
# update units
# Convert and replace
# Conversion factor
# km2_to_acres = 247.105381
# data["size (km2)"] = data["size (km2)"] * km2_to_acres
# data["fire_spread (km2/day)"] = data["fire_spread (km2/day)"] * km2_to_acres

# Rename the columns
# data.rename(columns={
    # "size (km2)": "size (acres)",
    # "fire_spread (km2/day)": "fire_spread (acres/day)"
# }, inplace=True)

In [ ]:
# Predictors and Targets
predictors = ['startdateseason', 'u10', 'v10', 'd2m', 't2m', 'msl', 'sp', 'lai_hv', 'lai_lv', 'tp', 'ssr']
target1 = 'size (acres)'
target2 = 'fire_spread (acres/day)'
target3 = 'duration'

In [ ]:
# Combine predictors and targets into a single list
# selected_columns = predictors + [target1, target2, target3]

# Select only these columns
# data = data[selected_columns]

In [ ]:
# view feature data
# data.head()

,unique_id,startdateseason,u10,v10,d2m,t2m,msl,sp,lai_hv,lai_lv,tp,ssr,size (acres),fire_spread (acres/day),duration
0,2020_22784,Winter,-3.701674,1.350839,278.64328,284.38715,102901.52,102447.46,2.988195,2.407728,0.000021,8017088.0,1218.62205,153.25470,8
8,2020_22960,Winter,0.051658,-4.528529,266.86755,275.61905,103046.79,102571.62,2.999217,2.415601,0.000000,13907856.0,1695.68910,242.24130,7
15,2020_23055,Winter,2.438791,1.001595,279.50827,284.98593,102126.12,101673.79,3.414510,2.855977,0.000003,10643344.0,7153.53390,650.09655,11
26,2020_23094,Winter,-5.122353,2.480373,287.34396,291.27878,102611.85,102358.31,2.270370,1.733630,0.000348,9271456.0,1537.49070,101.34585,15
41,2020_23265,Winter,-0.531692,1.984575,291.39870,293.45178,102482.13,102216.85,2.986109,2.869957,0.000510,8977760.0,6041.20140,301.56570,20


In [ ]:
# view feature data
# data.describe()

,u10,v10,d2m,t2m,msl,sp,lai_hv,lai_lv,tp,ssr,size (acres),fire_spread (acres/day),duration
count,32082.000000,32082.000000,32082.000000,32082.000000,32082.000000,32082.000000,32082.000000,32082.000000,32082.000000,3.208200e+04,32082.000000,32082.000000,32082.000000
mean,0.128195,0.341190,280.752792,290.498979,101587.466259,95404.544994,2.717593,1.764379,0.001590,1.680758e+07,4851.618310,400.545718,12.941774
std,1.832064,2.349705,8.882896,7.656374,614.691170,7389.700655,1.142001,0.716741,0.004740,5.577529e+06,12284.329982,991.401730,5.844891
min,-9.022443,-10.041690,247.640150,256.153400,99178.340000,70101.730000,0.000000,0.000000,0.000000,4.577280e+05,1006.042950,42.021450,1.000000
25%,-1.051346,-1.024666,274.147685,285.338530,101231.480000,91950.432500,2.030273,1.265003,0.000000,1.331969e+07,1324.911600,111.233250,8.000000
50%,0.174160,0.299106,280.678035,290.937680,101582.675000,97986.463000,2.593112,1.686295,0.000029,1.739407e+07,1960.177050,180.445050,12.000000
75%,1.194915,1.707381,287.887285,296.530053,101939.154250,100956.050000,3.549260,2.393430,0.000806,2.063463e+07,3497.667750,348.530850,17.000000
max,11.458554,10.673229,299.197050,311.581670,104212.430000,103518.164000,5.127843,3.819295,0.142104,2.985043e+07,199290.434400,33246.382500,31.000000


In [ ]:
# view feature data
# data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32082 entries, 0 to 32081
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   startdateseason          32082 non-null  object 
 1   u10                      32082 non-null  float64
 2   v10                      32082 non-null  float64
 3   d2m                      32082 non-null  float64
 4   t2m                      32082 non-null  float64
 5   msl                      32082 non-null  float64
 6   sp                       32082 non-null  float64
 7   lai_hv                   32082 non-null  float64
 8   lai_lv                   32082 non-null  float64
 9   tp                       32082 non-null  float64
 10  ssr                      32082 non-null  float64
 11  size (acres)             32082 non-null  float64
 12  fire_spread (acres/day)  32082 non-null  float64
 13  duration                 32082 non-null  int64  
dtypes: float64(12), int64(

In [ ]:
# split data
# train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# Write full, training, and test data to CSV files
# data.to_csv('../data/processed_data/full_wildfire_weather_2020_2024.csv', index=False)
# train_data.to_csv('../data/processed_data/training_wildfire_weather_2020_2024.csv', index=False)
# test_data.to_csv('../data/processed_data/test_wildfire_weather_2020_2024.csv', index=False)

In [ ]:
# read in data
train_data = pd.read_csv('../data/processed_data/training_wildfire_weather_2020_2024.csv')
test_data = pd.read_csv('../data/processed_data/test_wildfire_weather_2020_2024.csv')

In [ ]:
# Split the data
X_train = train_data[predictors]
y1_train = train_data[target1]
y2_train = train_data[target2]
y3_train = train_data[target3]


X_test = test_data[predictors]
y1_test = test_data[target1]
y2_test = test_data[target2]
y3_test = test_data[target3]

In [ ]:
# --- Step 4: One-hot encode the 'startdateseason' variable ---
X_train = pd.get_dummies(X_train, drop_first=True).astype(float)
X_test = pd.get_dummies(X_test, drop_first=True).astype(float)

# Ensure test set has same columns as train (fill missing with 0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
# Set seed and CV strategy
random_seed = 42
kf = KFold(n_splits=5, shuffle=True, random_state=random_seed)

In [ ]:
# Model 1: Predict size (km2)
# intialize the model
model1 = LinearRegression()

In [ ]:
# Cross-validation: MSE and RMSE
neg_mse1 = cross_val_score(model1, X_train, y1_train, cv=kf, scoring='neg_mean_squared_error')
rmse1 = np.sqrt(-neg_mse1)

In [ ]:
# Cross-validation: R²
r2_scores1 = cross_val_score(model1, X_train, y1_train, cv=kf, scoring='r2')

In [ ]:
# Statsmodels for coeffs/p-values and R² on full training data
X1_const = sm.add_constant(X_train)
ols1 = sm.OLS(y1_train, X1_const).fit()

print("\nCoefficients and p-values of the full model for predicting size (acres):")
print(ols1.summary())

print("\nEvaluation metrics for full model for predicting size (acres):")
# print(f"\nR² (full training data): {ols1.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols1.rsquared**0.5:.4f}")

print(f"5-fold CV Mean R²: {r2_scores1.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse1.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse1.mean():.4f}")



Coefficients and p-values of the full model for predicting size (acres):
                            OLS Regression Results                            
Dep. Variable:           size (acres)   R-squared:                       0.090
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     194.3
Date:                Fri, 18 Jul 2025   Prob (F-statistic):               0.00
Time:                        01:29:03   Log-Likelihood:            -2.7729e+05
No. Observations:               25665   AIC:                         5.546e+05
Df Residuals:                   25651   BIC:                         5.547e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------

In [ ]:
# Model 2: Predict fire_spread (acres/day)
model2 = LinearRegression()

In [ ]:
# Cross-validation: MSE and RMSE
neg_mse2 = cross_val_score(model2, X_train, y2_train, cv=kf, scoring='neg_mean_squared_error')
rmse2 = np.sqrt(-neg_mse2)

In [ ]:
# Cross-validation: R²
r2_scores2 = cross_val_score(model2, X_train, y2_train, cv=kf, scoring='r2')

In [ ]:
# Statsmodels summary for second model (full training data)
X2_const = sm.add_constant(X_train)
ols2 = sm.OLS(y2_train, X2_const).fit()

print("\nCoefficients and p-values of the full model for predicting fire spread (acres / day):")
print(ols2.summary())

print("\nEvaluation metrics for full model for predicting fire spread (acres / day):")
# print(f"\nR² (full training data): {ols2.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols2.rsquared**0.5:.4f}")

print(f"5-fold CV Mean R²: {r2_scores2.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse2.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse2.mean():.4f}")


Coefficients and p-values of the full model for predicting fire spread (acres / day):
                               OLS Regression Results                              
Dep. Variable:     fire_spread (acres/day)   R-squared:                       0.079
Model:                                 OLS   Adj. R-squared:                  0.078
Method:                      Least Squares   F-statistic:                     168.5
Date:                     Fri, 18 Jul 2025   Prob (F-statistic):               0.00
Time:                             01:29:03   Log-Likelihood:            -2.1291e+05
No. Observations:                    25665   AIC:                         4.259e+05
Df Residuals:                        25651   BIC:                         4.260e+05
Df Model:                               13                                         
Covariance Type:                 nonrobust                                         
                             coef    std err          t      P>|t|      [

In [ ]:
# Model 3: Predict duration (days)
model3 = LinearRegression()

In [ ]:
# Cross-validation: MSE and RMSE
neg_mse3 = cross_val_score(model3, X_train, y3_train, cv=kf, scoring='neg_mean_squared_error')
rmse3 = np.sqrt(-neg_mse3)

In [ ]:
# Cross-validation: R²
r2_scores3 = cross_val_score(model3, X_train, y3_train, cv=kf, scoring='r2')

In [ ]:
# Statsmodels summary for third model (full training data)
X3_const = sm.add_constant(X_train)
ols3 = sm.OLS(y3_train, X3_const).fit()

print("\nCoefficients and p-values of the full model for predicting duration (days):")
print(ols3.summary())

print("\nEvaluation metrics for full model for predicting duration (days):")
# print(f"\nR² (full training data): {ols3.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols3.rsquared**0.5:.4f}")

print(f"5-fold CV Mean R²: {r2_scores3.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse3.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse3.mean():.4f}")


Coefficients and p-values of the full model for predicting duration (days):
                            OLS Regression Results                            
Dep. Variable:               duration   R-squared:                       0.123
Model:                            OLS   Adj. R-squared:                  0.123
Method:                 Least Squares   F-statistic:                     276.8
Date:                Fri, 18 Jul 2025   Prob (F-statistic):               0.00
Time:                        01:29:04   Log-Likelihood:                -80007.
No. Observations:               25665   AIC:                         1.600e+05
Df Residuals:                   25651   BIC:                         1.602e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------

In [ ]:
def iterative_feature_elimination(X, y, alpha=0.05):
    X_const = sm.add_constant(X)
    ols = sm.OLS(y, X_const).fit()

    while True:
        pvalues = ols.pvalues.drop('const')
        max_p = pvalues.max()
        if max_p <= alpha:
            break  # All predictors significant enough

        # Remove predictor with max p-value
        worst_feature = pvalues.idxmax()
        print(f"Removing '{worst_feature}' with p-value {max_p:.4f}")
        X = X.drop(columns=[worst_feature])

        X_const = sm.add_constant(X)
        ols = sm.OLS(y, X_const).fit()

    return X, ols

# Set random seed and CV strategy (reuse kf)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# For Model 1
print("\n--- Iterative elimination of the model for predicting size (acres) ---")
X1_reduced, ols1_reduced = iterative_feature_elimination(X_train, y1_train)

model1_reduced = LinearRegression()
neg_mse1_reduced = cross_val_score(model1_reduced, X1_reduced, y1_train, cv=kf, scoring='neg_mean_squared_error')
rmse1_reduced = np.sqrt(-neg_mse1_reduced)
r2_scores1_reduced = cross_val_score(model1_reduced, X1_reduced, y1_train, cv=kf, scoring='r2')

print(f"\nReduced model for predicting size (acres) predictors: {list(X1_reduced.columns)}")
print("\nCoefficients and p-values after elimination (model for predicting size (acres)):")
print(ols1_reduced.summary())
print("\nEvaluation metrics for reduced model for predicting size (acres):")
# print(f"R² (full training data): {ols1_reduced.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols1_reduced.rsquared**0.5:.4f}")
print(f"5-fold CV Mean R²: {r2_scores1_reduced.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse1_reduced.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse1_reduced.mean():.4f}")

# For Model 2
print("\n--- Iterative elimination of model for predicting fire spread (acres / day) ---")
X2_reduced, ols2_reduced = iterative_feature_elimination(X_train, y2_train)

model2_reduced = LinearRegression()
neg_mse2_reduced = cross_val_score(model2_reduced, X2_reduced, y2_train, cv=kf, scoring='neg_mean_squared_error')
rmse2_reduced = np.sqrt(-neg_mse2_reduced)
r2_scores2_reduced = cross_val_score(model2_reduced, X2_reduced, y2_train, cv=kf, scoring='r2')

print(f"\nReduced Model 2 predictors: {list(X2_reduced.columns)}")
print("\nCoefficients and p-values for reduced model for predicting fire spread (acres / day):")
print(ols2_reduced.summary())
print("\nEvaluation metrics for reduced model for predicting fire spread (acres / day):")
# print(f"R² (full training data): {ols2_reduced.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols2_reduced.rsquared**0.5:.4f}")
print(f"5-fold CV Mean R²: {r2_scores2_reduced.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse2_reduced.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse2_reduced.mean():.4f}")

# For Model 3
print("\n--- Iterative elimination of model for duration (days) ---")
X3_reduced, ols3_reduced = iterative_feature_elimination(X_train, y3_train)

model3_reduced = LinearRegression()
neg_mse3_reduced = cross_val_score(model3_reduced, X3_reduced, y3_train, cv=kf, scoring='neg_mean_squared_error')
rmse3_reduced = np.sqrt(-neg_mse3_reduced)
r2_scores3_reduced = cross_val_score(model3_reduced, X3_reduced, y3_train, cv=kf, scoring='r2')

print(f"\nReduced Model 3 predictors: {list(X3_reduced.columns)}")
print("\nCoefficients and p-values for reduced model for duration (days):")
print(ols3_reduced.summary())
print("\nEvaluation metrics for reduced model for duration (days):")
# print(f"R² (full training data): {ols3_reduced.rsquared:.4f}")
# print(f"r correlation coefficient (full training data): {ols3_reduced.rsquared**0.5:.4f}")
print(f"5-fold CV Mean R²: {r2_scores3_reduced.mean():.4f}")
print(f"5-fold CV Mean MSE: {-neg_mse3_reduced.mean():.4f}")
print(f"5-fold CV Mean RMSE: {rmse3_reduced.mean():.4f}")



--- Iterative elimination of the model for predicting size (acres) ---
Removing 'sp' with p-value 0.9515
Removing 'v10' with p-value 0.5007

Reduced model for predicting size (acres) predictors: ['u10', 'd2m', 't2m', 'msl', 'lai_hv', 'lai_lv', 'tp', 'ssr', 'startdateseason_Spring', 'startdateseason_Summer', 'startdateseason_Winter']

Coefficients and p-values after elimination (model for predicting size (acres)):
                            OLS Regression Results                            
Dep. Variable:           size (acres)   R-squared:                       0.090
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     229.6
Date:                Fri, 18 Jul 2025   Prob (F-statistic):               0.00
Time:                        01:29:04   Log-Likelihood:            -2.7729e+05
No. Observations:               25665   AIC:                         5.546e+05
Df Residuals:                